# Policy Investigation — does the RL agent rediscover basic strategy, and where doesn't it?

This notebook investigates one trained run of the blackjack Monte Carlo agent, asking not
*whether* it matches the proven-optimal **basic strategy**, but **where it disagrees, and why**.

The agent learns to play from nothing but the win/loss reward at the end of each hand. Basic
strategy is the known optimal play, so it's our ground truth. The training run already saved —
per state cell — the agent's action vs basic's, the agent's value estimates (Q), the visit
counts, and a category label. We analyse that saved record; no retraining needed.

Working discipline: re-derive every headline before trusting it; separate real patterns from
base-rate artefacts (within-group checks); and distinguish *failed to learn* from *nothing to
learn* from *genuinely different*.

## 1. Load the evidence
`cells` is the per-cell policy diff (one row per state = player total × hard/soft × dealer
upcard); `qtable` is the raw learned Q-values and visit counts per (state, action).

In [1]:
import json
from pathlib import Path
import pandas as pd

# the 5M-episode run we're investigating
RUN_DIR = Path("../runs/20260617-032950_seed42_305b302")
record = json.loads((RUN_DIR / "record.json").read_text())

# the per-cell policy diff is the main investigation table; qtable is the raw learned values
cells = pd.DataFrame(record["diff"]["cells"])
qtable = pd.DataFrame(record["qtable"])

print("run:", record["run_id"])
print("eval:", record["eval"], "| diff thresholds:", record["diff"]["min_visits"], record["diff"]["ev_tol"])
print("cells:", len(cells), "| qtable rows:", len(qtable))
print("columns:", list(cells.columns))
cells.head()

run: 20260617-032950_seed42_305b302
eval: {'hands': 200000, 'seed': 0} | diff thresholds: 1000 0.02
cells: 280 | qtable rows: 820
columns: ['player_value', 'is_soft', 'dealer_upcard', 'visits', 'agent_action', 'basic_action', 'agent_q', 'basic_q', 'category']


,player_value,is_soft,dealer_upcard,visits,agent_action,basic_action,agent_q,basic_q,category
0,4,False,2,2125,hit,hit,-0.134726,-0.134726,agree
1,4,False,3,2259,hit,hit,-0.091650,-0.091650,agree
2,4,False,4,2205,hit,hit,-0.038000,-0.038000,agree
3,4,False,5,2161,hit,hit,0.013829,0.013829,agree
4,4,False,6,2195,hit,hit,-0.031065,-0.031065,agree


## 2. Re-derive the headline before trusting it
Recompute the category counts and agreement ourselves, then cross-check against the summary
saved at train time. If they don't match, that mismatch is the first thing to chase. Agreement
= fraction of cells where the agent matches basic; visit-weighted weights each cell by how
often it actually occurs.

In [2]:
# category breakdown, re-derived from the table
print(cells["category"].value_counts())

agree = cells["category"].eq("agree")
unweighted = agree.mean()
weighted = cells.loc[agree, "visits"].sum() / cells["visits"].sum()
print(f"\nagreement: {unweighted:.1%} unweighted, {weighted:.1%} visit-weighted")

# cross-check vs what was saved at train time
print("\nsaved:", record["diff"]["category_counts"],
      f"| {record['diff']['agreement_unweighted']:.3f} unweighted,"
      f" {record['diff']['agreement_weighted']:.3f} weighted")

category
agree                   259
genuine_disagreement     15
near_equal_ev             6
Name: count, dtype: int64

agreement: 92.5% unweighted, 94.0% visit-weighted

saved: {'agree': 259, 'near_equal_ev': 6, 'genuine_disagreement': 15} | 0.925 unweighted, 0.940 weighted


## 3. Look at the disagreements before theorising
Filter to `genuine_disagreement` — well-visited cells where the agent confidently chose a
different action than basic, with a meaningful value gap (the real mistakes, not ties or
coverage gaps). `ev_gap` = how far apart the agent thinks the two actions are. We read them
first, no stats yet.

In [3]:
genuine = cells[cells["category"] == "genuine_disagreement"].copy()
genuine["ev_gap"] = (genuine["agent_q"] - genuine["basic_q"]).abs()
genuine = genuine.sort_values("visits", ascending=False)
genuine[["player_value", "is_soft", "dealer_upcard", "visits",
         "agent_action", "basic_action", "agent_q", "basic_q", "ev_gap"]]

,player_value,is_soft,dealer_upcard,visits,agent_action,basic_action,agent_q,basic_q,ev_gap
168,16,False,10,166610,stand,hit,-0.540027,-5.610896e-01,0.021062
79,11,False,11,14996,double,hit,0.132559,8.620304e-02,0.046356
44,8,False,6,11757,double,hit,0.109421,7.058824e-02,0.038832
214,18,True,6,6859,stand,double,0.272528,1.739130e-01,0.098615
210,18,True,2,6835,double,stand,0.173039,1.021484e-01,0.070891
190,17,True,2,6726,double,hit,-0.013023,-4.479283e-02,0.031770
212,18,True,4,6191,stand,double,0.199025,9.405941e-02,0.104965
172,16,True,4,5616,hit,double,0.025030,-3.815029e-01,0.406533
174,16,True,6,5472,hit,double,0.086500,4.347826e-02,0.043022
134,14,True,6,4990,hit,double,0.125635,9.900990e-02,0.026625


## 4. Does the eyeballed pattern survive a base-rate check?
The rows look dominated by soft hands and doubling — but those cells might just be common. So
we compare the disagreement *rate within each group* (e.g. P(genuine | soft) vs P(genuine |
hard)), not raw counts. A far-higher within-group rate means real structure, not a base-rate
illusion.

In [4]:
g = cells.copy()
g["involves_double"] = g["agent_action"].eq("double") | g["basic_action"].eq("double")
gd = g[g["category"] == "genuine_disagreement"]

print("genuine disagreements:", len(gd))
print("  soft:           ", int(gd["is_soft"].sum()), "/", len(gd))
print("  involve double: ", int(gd["involves_double"].sum()), "/", len(gd))

# guard against base rates: disagreement RATE *within* each group, not raw counts
print("\nP(genuine_disagreement | group):")
for col in ["is_soft", "involves_double"]:
    rate = g.groupby(col).apply(
        lambda s: (s["category"] == "genuine_disagreement").mean(), include_groups=False
    )
    print(f"\n by {col}:")
    print(rate)

genuine disagreements: 15
  soft:            12 / 15
  involve double:  13 / 15

P(genuine_disagreement | group):

 by is_soft:
is_soft
False    0.016667
True     0.120000
dtype: float64

 by involves_double:
involves_double
False    0.008621
True     0.270833
dtype: float64


## 5. One pattern, or two mechanisms?
"Concentrates on doubling" is the *what*; the *why* could be two things with different fixes:
(a) the agent **under-explored** `double` (ε-greedy rarely tried it → noisy estimate), or (b)
it sampled `double` plenty and genuinely **over-values** it. We separate them by counting how
often each action was actually tried. The standard error on Q[double] ≈ 1.3/√(double count),
so a few-hundred-sample estimate is noisy on the scale of the gaps we saw.

In [5]:
# how often was each action actually tried at the genuine-disagreement cells?
keys = gd[["player_value", "is_soft", "dealer_upcard"]]
sub = qtable.merge(keys, on=["player_value", "is_soft", "dealer_upcard"])
pivot = sub.pivot_table(index=["player_value", "is_soft", "dealer_upcard"],
                        columns="action", values="n", fill_value=0)
pivot["total"] = pivot.sum(axis=1)
pivot["double_share"] = pivot["double"] / pivot["total"]
pivot.sort_values("double_share")

action                               double     hit     stand     total  \
player_value is_soft dealer_upcard                                        
16           False   10              3090.0  7489.0  156031.0  166610.0   
18           True    6                207.0   260.0    6392.0    6859.0   
                     11               148.0   287.0    4418.0    4853.0   
16           True    4                173.0  4994.0     449.0    5616.0   
13           True    6                150.0  4427.0     228.0    4805.0   
16           True    6                184.0  5052.0     236.0    5472.0   
13           True    5                171.0  4064.0     592.0    4827.0   
14           True    6                303.0  4529.0     158.0    4990.0   
18           True    4                404.0   250.0    5537.0    6191.0   
12           True    5                188.0  1788.0     160.0    2136.0   
                     6                249.0  1894.0      69.0    2212.0   
11           False   11              5507.0  8944.0     545.0   14996.0   
18           True    2               4080.0   288.0    2467.0    6835.0   
17           True    2               4300.0  1786.0     640.0    6726.0   
8            False   6              10254.0  1105.0     398.0   11757.0   

action                              double_share  
player_value is_soft dealer_upcard                
16           False   10                 0.018546  
18           True    6                  0.030179  
                     11                 0.030497  
16           True    4                  0.030805  
13           True    6                  0.031217  
16           True    6                  0.033626  
13           True    5                  0.035426  
14           True    6                  0.060721  
18           True    4                  0.065256  
12           True    5                  0.088015  
                     6                  0.112568  
11           False   11                 0.367231  
18           True    2                  0.596928  
17           True    2                  0.639310  
8            False   6                  0.872161